# Caterpillar Inc. (CAT) — Credit Analysis (FY2020–FY2024)

Bottom-up credit data built from primary SEC filings, separating Machinery, Energy & Transportation (ME&T) from the captive finance arm, Financial Products (Cat Financial + insurance). Each block is verified against the filing and checked with accounting identities — cells print `0` where an identity holds.

**Sources:** SEC Form 8-K Exhibit 99.1 earnings releases — FY20/FY21: 4Q21; FY22: 4Q23; FY23/FY24: 4Q24. Covenants: FY2022 10-K. Ratings: CAT FWP (May 2025).

*Analytical / educational; not investment advice.*

In [1]:
import pandas as pd

# CAT consolidated income statement, FY2020-FY2024 ($mm), as-reported
# Source: SEC 8-K ex-99.1 earnings releases
#   FY20/FY21 -> 4Q21 release    FY22 -> 4Q23 release    FY23/FY24 -> 4Q24 release
yrs = [2020, 2021, 2022, 2023, 2024]

inc = pd.DataFrame({
    "sales_met":     [39022, 48188, 56574, 63869, 61363],
    "rev_fp":        [2726,  2783,  2853,  3191,  3446],
    "revenue":       [41748, 50971, 59427, 67060, 64809],
    "cogs":          [29082, 35513, 41350, 42767, 40199], 
    "sga":           [4642,  5365,  5651,  6371,  6667],
    "rnd":           [1415,  1686,  1814,  2108,  2107],
    "int_fp":        [589,   455,   565,   1030,  1286],   # Financial Products interest, sits above OP
    "gw_imp":        [0,     0,     925,   0,     0],      # FY22 Rail/Longwall goodwill impairment
    "other_op":      [1467,  1074,  1218,  1818,  1478],
    "opcost":        [37195, 44093, 51523, 54094, 51737],
    "op_profit":     [4553,  6878,  7904,  12966, 13072],
    "int_met":       [514,   488,   443,   511,   512],    # ME&T interest, excl FP, sits below OP
    "other_inc":     [-44,   1814,  1291,  595,   813],
    "pretax":        [3995,  8204,  8752,  13050, 13373],
    "tax":           [1006,  1742,  2067,  2781,  2629],
    "profit_consol": [2989,  6462,  6685,  10269, 10744],
    "equity_aff":    [14,    31,    19,    63,    44],
    "profit_ca":     [3003,  6493,  6704,  10332, 10788],
    "nci":           [5,     4,     -1,    -3,    -4],      # NCI, subtracted to reach Profit
    "profit":        [2998,  6489,  6705,  10335, 10792],
}, index=yrs).T

chk = {
    "revenue = sales_met + rev_fp":           inc.loc["revenue"]   - inc.loc[["sales_met","rev_fp"]].sum(),
    "opcost = cogs+sga+rnd+int_fp+gw+other":  inc.loc["opcost"]    - inc.loc[["cogs","sga","rnd","int_fp","gw_imp","other_op"]].sum(),
    "op_profit = revenue - opcost":           inc.loc["op_profit"] - (inc.loc["revenue"] - inc.loc["opcost"]),
    "pretax = op_profit - int_met + other":   inc.loc["pretax"]    - (inc.loc["op_profit"] - inc.loc["int_met"] + inc.loc["other_inc"]),
    "profit = profit_ca - nci":               inc.loc["profit"]    - (inc.loc["profit_ca"] - inc.loc["nci"]),
}
for nm, c in chk.items():
    print(f"{int(c.abs().max()):>4}  {nm}")

   0  revenue = sales_met + rev_fp
   0  opcost = cogs+sga+rnd+int_fp+gw+other
   0  op_profit = revenue - opcost
   0  pretax = op_profit - int_met + other
   0  profit = profit_ca - nci


In [2]:
# CAT consolidated balance sheet, FY2020-FY2024 ($mm), period-end as-reported
# Source: SEC 8-K ex-99.1 condensed consolidated statement of financial position
#   FY20/FY21 -> 4Q21 release    FY22 -> 4Q23 release    FY23/FY24 -> 4Q24 release
bs = pd.DataFrame({
    "cash":     [9352,  9254,  7004,  6978,  6889],
    "inv":      [11402, 14038, 16270, 16565, 16827],
    "fin_recv": [9463,  8898,  9013,  9510,  9565],    # current finance receivables
    "tca":      [39464, 43455, 43785, 46949, 45682],
    "ppe":      [12401, 12090, 12028, 12680, 13361],
    "ta":       [78324, 82793, 81943, 87476, 87764],
    "st_met":   [10,    9,     3,     0,     0],        # ME&T short-term borrowings
    "ltw_met":  [1420,  45,    120,   1044,  46],       # ME&T LT debt due within 1yr
    "lta_met":  [9749,  9746,  9498,  8579,  8564],     # ME&T LT debt due after 1yr
    "st_fp":    [2005,  5395,  5954,  4643,  4393],     # Financial Products short-term borrowings
    "ltw_fp":   [7729,  6307,  5202,  7719,  6619],
    "lta_fp":   [16250, 16287, 16216, 15893, 18787],
    "tl":       [62946, 66277, 66052, 67973, 68270],
    "te":       [15378, 16516, 15891, 19503, 19494],
}, index=yrs).T

bs.loc["met_debt"] = bs.loc[["st_met","ltw_met","lta_met"]].sum()    # face, net of intercompany
bs.loc["fp_debt"]  = bs.loc[["st_fp","ltw_fp","lta_fp"]].sum()
bs.loc["tot_debt"] = bs.loc["met_debt"] + bs.loc["fp_debt"]

bs_chk = (bs.loc["ta"] - (bs.loc["tl"] + bs.loc["te"])).abs().max()
print(f"{int(bs_chk):>4}  ta = tl + te")
print("met_debt:", bs.loc["met_debt"].astype(int).tolist())
print("fp_debt :", bs.loc["fp_debt"].astype(int).tolist())
print("tot_debt:", bs.loc["tot_debt"].astype(int).tolist())

   0  ta = tl + te
met_debt: [11179, 9800, 9621, 9623, 8610]
fp_debt : [25984, 27989, 27372, 28255, 29799]
tot_debt: [37163, 37789, 36993, 37878, 38409]


In [3]:
# CAT consolidated cash flow + EBITDA bridge, FY2020-FY2024 ($mm), as-reported
# Source: SEC 8-K ex-99.1 condensed consolidated statement of cash flow
#   FY20/FY21 -> 4Q21 release    FY22 -> 4Q23 release    FY23/FY24 -> 4Q24 release
cf = pd.DataFrame({
    "dna":   [2432,  2352,  2219,  2144,  2153],     # depreciation & amortization
    "cfo":   [6327,  7198,  7766,  12885, 12035],    # net cash from operating activities
    "capex": [2115,  2472,  2599,  3092,  3215],     # capex incl equipment leased to others (ME&T + FP)
    "div":   [2243,  2332,  2440,  2563,  2646],     # dividends paid to shareholders
}, index=yrs).T

cf.loc["ebitda"] = inc.loc["op_profit"] + cf.loc["dna"]    # OP + D&A
cf.loc["ffo"]    = inc.loc["profit"]    + cf.loc["dna"]    # profit + D&A (FFO proxy)

print("ebitda:", cf.loc["ebitda"].astype(int).tolist())
print("ffo   :", cf.loc["ffo"].astype(int).tolist())

ebitda: [6985, 9230, 10123, 15110, 15225]
ffo   : [5430, 8841, 8924, 12479, 12945]


In [4]:
# CAT ME&T supplemental (Machinery, Energy & Transportation, ex-Financial Products), FY2020-FY2024 ($mm)
# Source: SEC 8-K ex-99.1 supplemental consolidating data (ME&T / FP columns)
#   FY20/FY21 -> 4Q21 release    FY22 -> 4Q23 release    FY23/FY24 -> 4Q24 release
met = pd.DataFrame({
    "op":     [4321,  6363,  7433,  12659, 13098],   # ME&T operating profit, consolidating basis (FY22 incl 925 gw impairment)
    "dna":    [1630,  1550,  1439,  1361,  1368],     # ME&T D&A
    "cash":   [8822,  8428,  6042,  6106,  6165],     # ME&T cash (not consolidated cash)
    "cfo":    [4054,  7177,  6358,  11688, 11437],    # ME&T operating cash flow
    "capex":  [976,   1088,  1279,  1624,  1952],     # ME&T capex, excl equipment leased to others
    "fp_dna": [802,   802,   780,   783,   785],      # FP D&A, for consolidated D&A cross-check
    "fp_div": [320,   850,   475,   425,   625],      # FP -> parent dividend (Cat Financial + Insurance)
}, index=yrs).T

met.loc["ebitda"] = met.loc["op"] + met.loc["dna"]                  # ME&T EBITDA
met.loc["nd"]     = bs.loc["met_debt"] - met.loc["cash"]            # net debt: face ME&T debt - ME&T cash
met.loc["fcf"]    = met.loc["cfo"] - met.loc["capex"]               # ME&T free cash flow
met.loc["lev"]    = (met.loc["nd"] / met.loc["ebitda"]).round(2)    # ME&T net leverage

dna_chk = ((met.loc["dna"] + met.loc["fp_dna"]) - cf.loc["dna"]).abs().max()
print(f"{int(dna_chk):>4}  ME&T D&A + FP D&A = consolidated D&A")
print("ME&T EBITDA   :", met.loc["ebitda"].astype(int).tolist())
print("ME&T net debt :", met.loc["nd"].astype(int).tolist())
print("ME&T ND/EBITDA:", met.loc["lev"].tolist())
print("ME&T FCF      :", met.loc["fcf"].astype(int).tolist())
print("FP->parent div:", met.loc["fp_div"].astype(int).tolist())

   0  ME&T D&A + FP D&A = consolidated D&A
ME&T EBITDA   : [5951, 7913, 8872, 14020, 14466]
ME&T net debt : [2357, 1372, 3579, 3517, 2445]
ME&T ND/EBITDA: [0.4, 0.17, 0.4, 0.25, 0.17]
ME&T FCF      : [3078, 6089, 5079, 10064, 9485]
FP->parent div: [320, 850, 475, 425, 625]


In [5]:
# CAT credit ratios, FY2020-FY2024 — derived from cells [1]-[4]
# Coverage uses int_met = consolidated "interest expense excluding Financial Products" (corporate interest);
#   int_fp is Cat Financial's cost of funds, classified in operating costs, so not corporate interest.
# ffo = profit + D&A proxy (cell [3]); ebitda_adj adds back the FY22 goodwill impairment.
ebitda     = cf.loc["ebitda"]
ebitda_adj = ebitda + inc.loc["gw_imp"]                            # normalize FY22 goodwill impairment
nd_consol  = bs.loc["tot_debt"] - bs.loc["cash"]

rat = pd.DataFrame({
    "nd_consol": nd_consol,
    "nde":       (nd_consol / ebitda).round(2),                    # consolidated net leverage
    "nde_adj":   (nd_consol / ebitda_adj).round(2),                # normalized (goodwill add-back)
    "cov":       (ebitda / inc.loc["int_met"]).round(1),           # consolidated EBITDA / ME&T interest
    "cov_met":   (met.loc["ebitda"] / inc.loc["int_met"]).round(1),# ME&T EBITDA / ME&T interest
    "ffo_debt":  (cf.loc["ffo"] / bs.loc["tot_debt"] * 100).round(1), # FFO proxy / total debt, %
    "met_lev":   met.loc["lev"],                                   # ME&T net leverage (cell [4])
}).T

fp_cash  = pd.Series([530, 826, 962, 872, 724], index=yrs)         # FP cash, supplemental FP column
cash_chk = ((met.loc["cash"] + fp_cash) - bs.loc["cash"]).abs().max()
print(f"{int(cash_chk):>4}  ME&T cash + FP cash = consolidated cash")
print(rat.to_string())

   0  ME&T cash + FP cash = consolidated cash
               2020      2021      2022      2023      2024
nd_consol  27811.00  28535.00  29989.00  30900.00  31520.00
nde            3.98      3.09      2.96      2.05      2.07
nde_adj        3.98      3.09      2.71      2.05      2.07
cov           13.60     18.90     22.90     29.60     29.70
cov_met       11.60     16.20     20.00     27.40     28.30
ffo_debt      14.60     23.40     24.10     32.90     33.70
met_lev        0.40      0.17      0.40      0.25      0.17


In [6]:
# CAT Financial Products credit quality (Cat Financial + Insurance), FY2020-FY2024
# Sources: asset quality -> 4Q earnings releases (FP section); FP debt -> cell [2];
#          FP equity -> Q4 supplemental BS (FP column); FY22 covenants -> FY2022 10-K (cat-20221231.htm)
fp = pd.DataFrame({
    "allow_pct":   [1.77, 1.22, 1.29, 1.18, 0.91],   # allowance for credit losses, % of finance receivables
    "pastdue_pct": [3.49, 1.95, 1.89, 1.79, 1.56],   # past dues, %
    "writeoff":    [222,  205,  46,   65,   115],     # write-offs net of recoveries, $mm
    "fp_equity":   [4645, 4257, 4032, 4399, 4246],    # FP segment equity, supplemental BS FP column
}, index=yrs).T
fp.loc["fp_debt"] = bs.loc["fp_debt"]                              # from cell [2]
fp.loc["lev"]     = (fp.loc["fp_debt"] / fp.loc["fp_equity"]).round(2)   # FP debt / equity

# Cat Financial covenant compliance at 12/31/22 (FY2022 10-K); $4.62bn facility, expires Sep-2027
cov_fy22 = {"int_cov": 2.36, "int_cov_floor": 1.15,       # interest coverage vs minimum
            "leverage": 7.05,                             # six-month covenant leverage ratio
            "net_worth": 15.93, "net_worth_floor": 9.00}  # consolidated net worth ($bn) vs minimum

print("FP debt/equity :", fp.loc["lev"].tolist())
print("allowance %    :", fp.loc["allow_pct"].tolist())
print("past dues %    :", fp.loc["pastdue_pct"].tolist())
print("FY22 covenant  :", cov_fy22)






FP debt/equity : [5.59, 6.57, 6.79, 6.42, 7.02]
allowance %    : [1.77, 1.22, 1.29, 1.18, 0.91]
past dues %    : [3.49, 1.95, 1.89, 1.79, 1.56]
FY22 covenant  : {'int_cov': 2.36, 'int_cov_floor': 1.15, 'leverage': 7.05, 'net_worth': 15.93, 'net_worth_floor': 9.0}
